#### **Step 5 - The Retriever Class**

In [1]:
from sparse_index import SparseIndex
from dense_index import DenseIndex 
from fusion import reciprocal_rank_fusion

# combines all retrieval methods into a single object with a consistent interface
class HybridRetriever:
    def __init__(self, chunks):
        self.chunks = chunks 
        self.sparse_index = SparseIndex(chunks)
        self.dense_index = DenseIndex(chunks)

    # individual thin wrappers for the standalone search methods 
    # this allows us to perform individual component searches on the chunks 
    def search_sparse(self, query, k=5):
        return self.sparse_index.search(query, k)
        
    def search_dense(self, query, k=5):
        return self.dense_index.search(query, k)

    # one giant function to call the individual search methods and create a fused ranking
    def search_hybrid(self, query, k=5, overfetch=10):
        # the idea is we want to overfetch a certain number chunks and then take the top k from them
        sparse_result = self.search_sparse(query, k=overfetch)
        dense_result = self.search_dense(query, k=overfetch)
        result = reciprocal_rank_fusion([sparse_result, dense_result], k=60)
        return result[:k]

    def get_texts(self, results):
        # pull ot the chunk texts from the results - used for the next part which is the pipeline
        return [chunk["text"] for chunk, _score in results]

**`HybridRetriever` Intuitive Summary**

- The idea is to combine all the retrieval methods into a single convenient class 

- We want to first instantiate all the retrieval methods which we will use in the the later search methods as well as instantiate the chunks 

- We want to create individual search methods for each of the retrieval methods we use - this will allow us to perform component searches on the chunks as well as 1 giant search combining all retrieval methods 

- `search_hybrid` aims to take all the outputs of each of the individual component retrieval methods and pass them through `reciprocal_rank_fusion` to create a combined score from all the retrieval methods - call the thin wrapper functions for each individual retrieval method in this single method to combine all the retrieval outputs

- `get_texts` is just a convenient function used to return a list of the text in each chunk for the top k results - this is useful for the next step which is the RAG pipeline

**Note: We could have searched with each retrieval method in `search_hybrid` but the individual retrieval functions allow us to modularise retrievals into functions so we can use them individually or combine them**

**Testing**

In [6]:
from sparse_index import SparseIndex
from dense_index import DenseIndex
from fusion import reciprocal_rank_fusion
from retriever import HybridRetriever
from chunking import load_documents, chunk_documents

docs = load_documents("../data")
chunks = chunk_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

retriever = HybridRetriever(chunks)

query = "How many championships has Faker won?"

for method_name, method_fn in [
    ("sparse", retriever.search_sparse),
    ("dense", retriever.search_dense),
    ("hybrid", retriever.search_hybrid),
]:
    print(f"\n--- {method_name} ---")
    for chunk, score in method_fn(query, k=4):
        print(f"  {score:.4f} | {chunk['chunk_id']} | {chunk['text']}...")

1 documents -> 22 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- sparse ---
  4.7361 | faker.txt_chunk3 |  player in the history of the game.

Outside of Worlds, Faker has won the Mid-Season Invitational twice, in 2016 and 2017, along with a large number of LCK titles in the Korean domestic league. He is also known for a wide champion pool. Over the years he has been associated with LeBlanc, Ryze, Azir, Syndra, Orianna and many others, and he was among the first mid laners to show how flexible a single role could be.

His playstyle is defined by mechanical precision, calm decision making and strong map awareness. Teammates and coaches often say his greatest strength is not his hands but his reading of the game: he seems to know where the enemy jungler...
  3.1061 | faker.txt_chunk17 |  constantly, which is one reason longevity is considered such an achievement.

Championships in traditional sports are decided by long seasons and playoffs, and esports has borrowed the same structure. In football, the championships of the major leagues attract bi